# Site Selection — Multi-Factor Scoring Model

A specialty coffee chain is planning seven new Bay Area storefronts.
Real estate is scarce and long-leased — picking wrong costs eight
figures over a decade. This notebook builds the quantitative shortlist
the real estate committee will argue over:

1. **Candidates** — seven broker-sourced addresses across SF, the East
   Bay, and the Peninsula.
2. **Reach** — a 10-minute drive-time isochrone per candidate via
   `ST_Isochrone` — that's the trade area.
3. **Demographics** — area-weighted catchment population + median
   household income from U.S. Census ZCTA polygons intersected with each
   trade area.
4. **Competition** — count of existing coffee shops inside each trade
   area from the Foursquare POI catalog.
5. **Score** — weighted composite: population (40%) + income (35%) +
   whitespace (25%) — whitespace is inverse competitor density.
6. **Shortlist** — ranked table + GeoJSON feed for the committee's map.

> **Demographics note.** ACS block-group values aren't in the Wherobots
> public catalog, so this notebook synthesizes per-ZCTA population and
> median income deterministically from the ZCTA ID. The *method* —
> area-weighted intersection of demographics polygons with isochrones —
> is the production pattern; swap in your ACS / Claritas / PolicyMap
> table and the rest of the pipeline is unchanged.

## 1. Setup

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import json

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

DRIVE_MINUTES = 10

## 2. Candidate Sites

Seven broker-sourced Bay Area storefronts — a mix of urban, suburban,
and commuter locations so the scoring model has to actually
discriminate.

In [ ]:
candidates = [
    # site_id, label,                   lon,       lat,     asking_rent_psf
    ("CAND-01", "SF Hayes Valley",     -122.4244, 37.7763, 96.0),
    ("CAND-02", "SF Mission / Valencia", -122.4214, 37.7601, 84.0),
    ("CAND-03", "Oakland Rockridge",   -122.2515, 37.8436, 72.0),
    ("CAND-04", "Berkeley Downtown",   -122.2712, 37.8715, 78.0),
    ("CAND-05", "Palo Alto University", -122.1587, 37.4448, 112.0),
    ("CAND-06", "San Jose Santana Row", -121.9489, 37.3207, 92.0),
    ("CAND-07", "Walnut Creek Broadway",-122.0652, 37.9000, 68.0),
]

cand_schema = StructType([
    StructField("site_id",         StringType()),
    StructField("label",           StringType()),
    StructField("lon",             DoubleType()),
    StructField("lat",             DoubleType()),
    StructField("asking_rent_psf", DoubleType()),
])

candidates_df = sedona.createDataFrame(candidates, cand_schema) \
    .withColumn("geometry", f.expr("ST_Point(lon, lat)"))
candidates_df.createOrReplaceTempView("candidates")
candidates_df.show(truncate=False)

## 3. Generate 10-Minute Drive-Time Isochrones

`ST_Isochrone(point, minutes, mode, reverse)` returns the polygon
reachable within `minutes` by `mode`. One call per candidate; Spark
parallelizes.

In [ ]:
isochrones_df = sedona.sql(f"""
    SELECT
        site_id,
        label,
        lon, lat,
        asking_rent_psf,
        geometry AS site_point,
        ST_Isochrone(geometry, {DRIVE_MINUTES}, 'car', false) AS isochrone
    FROM candidates
""").cache()
isochrones_df.createOrReplaceTempView("isochrones")

sedona.sql("""
    SELECT
        site_id, label,
        ROUND(
            ST_Area(ST_Transform(isochrone, 'EPSG:4326', 'EPSG:3857')) / 1e6, 2
        ) AS trade_area_sq_km
    FROM isochrones
    ORDER BY trade_area_sq_km DESC
""").show(truncate=False)

## 4. Prepare Demographics — ZCTA Polygons + Synthesized Values

Pull U.S. Census ZCTA polygons intersecting a Bay-Area bbox and attach
deterministic population / median-income values keyed off the ZCTA ID.
In production replace the hash-derived values with your ACS or
commercial demographics table.

In [ ]:
bay_zips_df = sedona.sql("""
    SELECT
        ZCTA5CE10 AS zcta5,
        geometry,
        -- Deterministic demo values: population 15k-75k, income 55k-185k
        CAST(15000 + (ABS(HASH(ZCTA5CE10))          % 60000) AS INT) AS population,
        CAST(55000 + (ABS(HASH(CONCAT('i', ZCTA5CE10))) % 130000) AS INT) AS median_income
    FROM wherobots_pro_data.us_census.zipcode
    WHERE ST_Intersects(
        geometry,
        ST_MakeEnvelope(-123.0, 37.1, -121.5, 38.2, 4326)
    )
""").cache()
bay_zips_df.createOrReplaceTempView("bay_zips")

print(f"ZCTAs in Bay Area bbox: {bay_zips_df.count()}")
bay_zips_df.select("zcta5", "population", "median_income") \
           .orderBy(f.col("population").desc()).show(5, truncate=False)

### Area-weighted catchment demographics

For each isochrone, compute total reachable population and
population-weighted median income by intersecting with ZCTA polygons
and prorating by intersected-area fraction.

In [ ]:
demo_df = sedona.sql("""
    WITH intersections AS (
        SELECT
            i.site_id,
            i.label,
            z.zcta5,
            z.population,
            z.median_income,
            ST_Area(ST_Intersection(i.isochrone, z.geometry))
                / NULLIF(ST_Area(z.geometry), 0) AS area_fraction
        FROM isochrones i
        JOIN bay_zips z
          ON ST_Intersects(i.isochrone, z.geometry)
    )
    SELECT
        site_id,
        label,
        COUNT(*)                                     AS zip_count,
        CAST(ROUND(SUM(population * area_fraction)) AS INT)
                                                     AS catchment_population,
        CAST(ROUND(
            SUM(population * area_fraction * median_income)
            / NULLIF(SUM(population * area_fraction), 0)
        ) AS INT)                                    AS weighted_median_income
    FROM intersections
    GROUP BY site_id, label
""").cache()
demo_df.createOrReplaceTempView("site_demographics")
demo_df.orderBy(f.col("catchment_population").desc()).show(truncate=False)

## 5. Competitor Density — Foursquare Coffee Shops

Pull every coffee-shop / café POI inside the Bay-Area bbox from the
Foursquare places catalog, then spatial-join to each isochrone. Count =
direct competitors inside the trade area.

In [ ]:
competitors_df = sedona.sql("""
    SELECT
        fsq_place_id, name, geom
    FROM wherobots_open_data.foursquare.places
    WHERE longitude BETWEEN -123.0 AND -121.5
      AND latitude  BETWEEN 37.1  AND 38.2
      AND exists(fsq_category_labels, x -> x LIKE '%Coffee%')
      AND date_closed IS NULL
""").cache()
competitors_df.createOrReplaceTempView("bay_coffee_shops")
print(f"Bay-area coffee shops in Foursquare: {competitors_df.count():,}")

comp_counts_df = sedona.sql("""
    SELECT
        i.site_id,
        i.label,
        COUNT(c.fsq_place_id) AS competitor_count
    FROM isochrones i
    LEFT JOIN bay_coffee_shops c
      ON ST_Intersects(i.isochrone, c.geom)
    GROUP BY i.site_id, i.label
""").cache()
comp_counts_df.createOrReplaceTempView("site_competitors")
comp_counts_df.orderBy(f.col("competitor_count").desc()).show(truncate=False)

## 6. Compose the Multi-Factor Score

Join the demographic and competitor signals onto the candidates,
min-max normalize each factor to [0, 1] across the candidate pool, and
combine:

$$ \text{score} = 0.40 \cdot \widehat{pop} + 0.35 \cdot \widehat{income} + 0.25 \cdot (1 - \widehat{competitors}) $$

The last term — *whitespace* — rewards low competitor density. Weights
are the real-estate committee's knobs; easy to adjust.

In [ ]:
scored_df = sedona.sql("""
    WITH joined AS (
        SELECT
            c.site_id,
            c.label,
            c.lon, c.lat,
            c.asking_rent_psf,
            d.catchment_population,
            d.weighted_median_income,
            COALESCE(cc.competitor_count, 0) AS competitor_count
        FROM candidates c
        LEFT JOIN site_demographics d USING (site_id)
        LEFT JOIN site_competitors  cc USING (site_id)
    ),
    ranges AS (
        SELECT
            MIN(catchment_population)   AS pop_min,
            MAX(catchment_population)   AS pop_max,
            MIN(weighted_median_income) AS inc_min,
            MAX(weighted_median_income) AS inc_max,
            MIN(competitor_count)       AS comp_min,
            MAX(competitor_count)       AS comp_max
        FROM joined
    )
    SELECT
        j.site_id, j.label, j.lon, j.lat,
        j.asking_rent_psf,
        j.catchment_population,
        j.weighted_median_income,
        j.competitor_count,
        ROUND((j.catchment_population   - r.pop_min)  / NULLIF(r.pop_max  - r.pop_min,  0), 3) AS pop_norm,
        ROUND((j.weighted_median_income - r.inc_min)  / NULLIF(r.inc_max  - r.inc_min,  0), 3) AS inc_norm,
        ROUND((j.competitor_count       - r.comp_min) / NULLIF(r.comp_max - r.comp_min, 0), 3) AS comp_norm,
        ROUND(
            0.40 * ((j.catchment_population   - r.pop_min)  / NULLIF(r.pop_max  - r.pop_min,  0)) +
            0.35 * ((j.weighted_median_income - r.inc_min)  / NULLIF(r.inc_max  - r.inc_min,  0)) +
            0.25 * (1.0 - (j.competitor_count - r.comp_min) / NULLIF(r.comp_max - r.comp_min, 0)),
            4
        ) AS composite_score
    FROM joined j CROSS JOIN ranges r
""").cache()
scored_df.createOrReplaceTempView("scored_candidates")

scored_df.orderBy(f.col("composite_score").desc()) \
         .select("site_id", "label", "catchment_population",
                 "weighted_median_income", "competitor_count",
                 "pop_norm", "inc_norm", "comp_norm",
                 "composite_score") \
         .show(truncate=False)

## 7. The Shortlist

Rank and add a rent-adjusted score — composite divided by rent PSF —
for a capital-efficiency view the CFO will ask for.

In [ ]:
shortlist_df = sedona.sql("""
    SELECT
        RANK() OVER (ORDER BY composite_score DESC) AS rank,
        site_id, label,
        catchment_population,
        weighted_median_income,
        competitor_count,
        asking_rent_psf,
        composite_score,
        ROUND(composite_score / asking_rent_psf * 100, 4)
            AS rent_efficiency_score
    FROM scored_candidates
    ORDER BY composite_score DESC
""")
shortlist_df.show(truncate=False)

## 8. Export — GeoJSON for the Committee's Map

One FeatureCollection with each candidate as a point feature carrying
all the scoring fields; a second file for the isochrones so the map can
shade the trade areas.

In [ ]:
# Candidate points with scores
pts_rows = sedona.sql("""
    SELECT
        s.site_id, s.label, s.asking_rent_psf,
        s.catchment_population, s.weighted_median_income,
        s.competitor_count, s.composite_score,
        ST_AsGeoJSON(ST_Point(s.lon, s.lat)) AS geojson
    FROM scored_candidates s
""").collect()

point_features = [
    {
        "type": "Feature",
        "properties": {
            "site_id":                r.site_id,
            "label":                  r.label,
            "asking_rent_psf":        r.asking_rent_psf,
            "catchment_population":   int(r.catchment_population or 0),
            "weighted_median_income": int(r.weighted_median_income or 0),
            "competitor_count":       int(r.competitor_count or 0),
            "composite_score":        float(r.composite_score or 0),
        },
        "geometry": json.loads(r.geojson),
    }
    for r in pts_rows
]
pts_path = "/tmp/site_selection_candidates.geojson"
with open(pts_path, "w") as fh:
    json.dump({"type": "FeatureCollection", "features": point_features}, fh, indent=2)
print(f"Wrote {pts_path} ({len(point_features)} candidates)")

# Trade-area isochrones
iso_rows = sedona.sql("""
    SELECT site_id, label, ST_AsGeoJSON(isochrone) AS geojson
    FROM isochrones
""").collect()

iso_features = [
    {
        "type": "Feature",
        "properties": {"site_id": r.site_id, "label": r.label,
                       "drive_minutes": DRIVE_MINUTES},
        "geometry": json.loads(r.geojson),
    }
    for r in iso_rows
]
iso_path = "/tmp/site_selection_trade_areas.geojson"
with open(iso_path, "w") as fh:
    json.dump({"type": "FeatureCollection", "features": iso_features}, fh, indent=2)
print(f"Wrote {iso_path} ({len(iso_features)} trade areas)")

## 9. Preview on a Map

In [ ]:
viz = SedonaKepler.create_map()
SedonaKepler.add_df(
    viz,
    isochrones_df.selectExpr("site_id", "label", "isochrone AS geometry"),
    name="Trade Areas (10 min)"
)
SedonaKepler.add_df(
    viz,
    scored_df.select("site_id", "label", "catchment_population",
                     "competitor_count", "composite_score",
                     f.expr("ST_Point(lon, lat)").alias("geometry")),
    name="Candidates"
)
viz

---

## Scaling the model

| Knob | How to change |
|---|---|
| Score weights | Adjust the three constants in the SQL (`0.40 / 0.35 / 0.25`) |
| Trade-area size | Change `DRIVE_MINUTES` at the top or switch to `'foot'` mode |
| Competitor category | Replace the `LIKE '%Coffee%'` filter (e.g., `%Quick Service%` for QSR) |
| More candidates | Extend the `candidates` list — `ST_Isochrone` parallelizes |
| Real demographics | Swap `bay_zips` for an ACS / Claritas / PolicyMap table joined on ZCTA |
| Additional factors | Add foot-traffic (SafeGraph), transit-stop density (OSM), parking (Overture base) — same join pattern |

## Outputs

| File | Contents |
|---|---|
| `/tmp/site_selection_candidates.geojson` | Point features, one per site, with full scoring fields |
| `/tmp/site_selection_trade_areas.geojson` | 10-min drive-time polygons for the map background |